# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print('total revenue:', total_revenue)
print('total units:', total_units)

total revenue: 8520.0
total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()
by_category['share_pct'] = (100 * by_category['revenue'] / total_revenue).round(1)
by_category

,category,revenue,share_pct
0,Food,4293.0,50.4
1,Merch,1771.5,20.8
2,Drink,1554.0,18.2
3,RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_summary = df.groupby('vendor_id')['revenue'].agg(['mean', 'count']).sort_values('mean', ascending=False)
vendor_summary

,mean,count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_share = 100 * df.loc[df['category'] == 'Merch', 'revenue'].sum() / total_revenue
print('Merch share of total revenue:', round(merch_share, 1), '%')

Merch share of total revenue: 20.8 %


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
print('rows:', len(joined), '| revenue:', joined['revenue'].sum())

unmatched = df.loc[~df['vendor_id'].isin(vendor_names['vendor_id']), 'vendor_id'].unique()
unmatched_revenue = df.loc[df['vendor_id'].isin(unmatched), 'revenue'].sum()
print('unmatched vendor:', unmatched, '| revenue attached:', unmatched_revenue)

rows: 400 | revenue: 8520.0
unmatched vendor: ['V-18'] | revenue attached: 2349.0


**The unmatched vendor, and what I did about it:** V-18 is not in the vendor_names lookup, for $2,349 (about 27.6% of total revenue). That's too big a share to drop, so I left it in the merged frame with a NaN vendor_name via the left join rather than filter it out. This way you can see it in any downstream report, rather than having it disappear silently.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot = df.pivot_table(
    index='vendor_id', columns='category', values='revenue',
    aggfunc='sum', fill_value=0, margins=True, margins_name='Total'
)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) Food is by far the game’s biggest revenue driver, comprising just over half of all sales ($4,293, 50.4%), so vendors should keep Food stock deep and well-staffed above all else. RainGear, while it usually gets plenty of attention, was just 10.6% ($901.50) of revenue this game - worth watching if weather turns, but shouldn't be the top staffing priority from this data alone.

b) The weakest part of this report is Q5. Vendor V-18, which is $2,349 or more than a quarter of total revenue, has no name in the lookup table so any breakdown at the vendor level based on vendor_name instead of vendor_id will undercount or mislabel silently by more than a quarter of the business. Vendor-name based summaries must be considered incomplete until that vendor can be found and added to the lookup.